> **Reused upstream sample; local workshop adaptation.** Source: [Azure-Samples/azure-ai-content-understanding-python](https://github.com/Azure-Samples/azure-ai-content-understanding-python/blob/19f1a9d68ef5de66abef056564f32e4bc49f4067/notebooks/field_extraction.ipynb).
> Original commit: `19f1a9d68ef5de66abef056564f32e4bc49f4067`. MIT notice: [LICENSE](../LICENSE).
> Local changes: repo .venv/.env and Entra-only auth; read-only defaults; existing configured completion model; compact display previews with complete JSON saved by upstream helpers; explicit opt-in sample cleanup. Analysis calls incur Azure usage. No new processing framework.


# Extract Custom Fields from Your File

This notebook demonstrates how to use analyzers to extract custom fields from your input files.

Content Understanding provides **extensive prebuilt analyzers** ready to use without training. Always start with prebuilt analyzers before building custom solutions.

## Prerequisites
1. Ensure Azure AI service is configured following [steps](../README.md#configure-azure-ai-service-resource)
2. Install the required packages to run the sample.

Use the repository's existing **.venv**. This upstream install cell is intentionally not executed; install only a specifically missing package in a visible terminal.


## Use the existing upstream REST client
These educational demos retain the upstream REST wrapper and analysis/schema code. This local setup reads the root .env, uses the shared .venv and Microsoft Entra ID only. The claims app continues to use the official SDK.


In [11]:
from datetime import datetime
from pathlib import Path
import logging
import json
import os
import sys
import time
import asyncio
from typing import Any, Optional
from dotenv import find_dotenv, load_dotenv

ENV_FILE = Path(find_dotenv(usecwd=True, raise_error_if_not_found=True))
load_dotenv(ENV_FILE, override=True)
NOTEBOOK_DIR = ENV_FILE.parent / "demo" / "content-understanding" / "notebooks"
os.chdir(NOTEBOOK_DIR)
sys.path.insert(0, str(NOTEBOOK_DIR.parent / "python"))
from content_understanding_client import AzureContentUnderstandingClient
from extension.sample_helper import save_json_to_file
from azure.identity import DefaultAzureCredential

if Path(sys.executable).resolve() != (ENV_FILE.parent / ".venv" / "Scripts" / "python.exe").resolve():
    raise RuntimeError("Select the repository .venv kernel.")
logging.basicConfig(level=logging.WARNING)
AZURE_AI_ENDPOINT = os.environ["APP_CONTENT_UNDERSTANDING_ENDPOINT"]
API_VERSION = os.getenv("CU_API_VERSION", "2025-11-01")
credential = DefaultAzureCredential()
def token_provider():
    return credential.get_token("https://cognitiveservices.azure.com/.default").token

client = AzureContentUnderstandingClient(
    endpoint=AZURE_AI_ENDPOINT, api_version=API_VERSION,
    token_provider=token_provider,
    x_ms_useragent="azure-ai-content-understanding-python-sample-ga",
)
RUN_SAMPLE_CLEANUP = False
print("Official upstream REST sample |", API_VERSION, "| Microsoft Entra ID")
print("Shared repo environment:", ENV_FILE.name, "| Kernel:", Path(sys.executable).relative_to(ENV_FILE.parent))


Official upstream REST sample | 2025-11-01 | Microsoft Entra ID
Shared repo environment: .env | Kernel: .venv\Scripts\python.exe


## Inspect existing model configuration (read-only)
The upstream default-update example is replaced here with a read-only check. No models are deployed and no shared defaults are changed. Custom examples use a model already mapped to the configured completion deployment.


In [12]:
defaults = client.get_defaults()
model_deployments = defaults.get("modelDeployments", {})
print("Existing resource model mappings (read-only):", model_deployments)
completion_deployment = os.environ["APP_CU_COMPLETION_DEPLOYMENT"]
completion_models = [model for model, deployment in model_deployments.items() if deployment == completion_deployment]
if not completion_models:
    raise ValueError("The configured completion deployment is not mapped in this account. Review resource defaults; this notebook will not patch them.")
COMPLETION_MODEL = completion_deployment if completion_deployment in completion_models else completion_models[0]
print("Custom sample completion model:", COMPLETION_MODEL, "| Existing deployment:", completion_deployment)


Existing resource model mappings (read-only): {'gpt-5.2': 'gpt-5.2', 'text-embedding-3-large': 'text-embedding-3-large', 'prebuilt-analyzer-completion': 'gpt-5.2', 'prebuilt-analyzer-completion-mini': 'gpt-5.2', 'prebuilt-analyzer-embedding': 'text-embedding-3-large'}
Custom sample completion model: gpt-5.2 | Existing deployment: gpt-5.2


# Part 1: Using Prebuilt Analyzers (Recommended Starting Point)

## Why Start with Prebuilt Analyzers?

Azure AI Content Understanding provides **70+ production-ready prebuilt analyzers** that cover common scenarios across finance, healthcare, legal, tax, and business domains. These analyzers are:

- **Immediately Available** - No training, configuration, or customization needed  
- **Battle-Tested** - Built on rich knowledge bases of thousands of real-world document examples  
- **Continuously Improved** - Regularly updated by Microsoft to handle document variations  
- **Cost-Effective** - Save development time and resources by using proven solutions  
- **Comprehensive Coverage** - Extensive support for Financial documents (invoices, receipts, bank statements, credit cards), Identity documents (passports, driver licenses, ID cards, health insurance), Tax documents (40+ US tax forms including 1040, W-2, 1099 variants), Mortgage documents (applications, appraisals, disclosures), Business documents (contracts, purchase orders, procurement), and many more specialized scenarios

> **Best Practice**: Always explore prebuilt analyzers first. Build custom analyzers only when prebuilt options don't meet your specific requirements.

### Complete List of Prebuilt Analyzer Categories

**Content Extraction & RAG**
- `prebuilt-read`, `prebuilt-layout` - OCR and layout analysis
- `prebuilt-documentSearch`, `prebuilt-imageSearch`, `prebuilt-audioSearch`, `prebuilt-videoSearch` - RAG-optimized

**Financial Documents**
- `prebuilt-invoice`, `prebuilt-receipt`, `prebuilt-creditCard`, `prebuilt-bankStatement.us`, `prebuilt-check.us`, `prebuilt-creditMemo`

**Identity & Healthcare**  
- `prebuilt-idDocument`, `prebuilt-idDocument.passport`, `prebuilt-healthInsuranceCard.us`

**Tax Documents (US)**
- 40+ tax form analyzers including `prebuilt-tax.us.1040`, `prebuilt-tax.us.w2`, all 1099 variants, 1098 series, and more

**Mortgage Documents (US)**
- `prebuilt-mortgage.us.1003`, `prebuilt-mortgage.us.1004`, `prebuilt-mortgage.us.1005`, `prebuilt-mortgage.us.closingDisclosure`

**Legal & Business**
- `prebuilt-contract`, `prebuilt-procurement`, `prebuilt-purchaseOrder`, `prebuilt-marriageCertificate.us`

**Other Specialized**
- `prebuilt-utilityBill`, `prebuilt-payStub.us`, and more

> **Learn More**: [Complete Prebuilt Analyzers Documentation](https://learn.microsoft.com/en-us/azure/ai-services/content-understanding/concepts/prebuilt-analyzers)

## Build Custom Analyzers (When Needed)

Create custom analyzers only when prebuilt ones don't meet your needs:
- Extract fields specific to your business
- Process proprietary document types
- Customize extraction logic for unique requirements

**This notebook demonstrates both approaches:**
1. **Part 1**: Using prebuilt analyzers (receipts, invoices)
2. **Part 2**: Creating custom analyzers when prebuilt options aren't sufficient

## 1. Invoice Field Extraction with Prebuilt Analyzer

Let's demonstrate using `prebuilt-invoice` to extract structured data from an invoice PDF. This analyzer automatically identifies vendor information, invoice numbers, dates, line items, totals, taxes, and payment details without any configuration.


In [3]:
sample_file_path = '../data/invoice.pdf'
invoice_analyzer_id = "prebuilt-invoice"

print(f"Analyzing {sample_file_path} with {invoice_analyzer_id}...")

analysis_response = client.begin_analyze_binary(
    analyzer_id=invoice_analyzer_id,
    file_location=sample_file_path,
)

# Wait for analysis completion
print(f"Waiting for document analysis to complete...")
analysis_result = client.poll_result(analysis_response)
print(f"Document analysis completed successfully!")



Analyzing ../data/invoice.pdf with prebuilt-invoice...
Waiting for document analysis to complete...
Document analysis completed successfully!


**Invoice Analysis Results**

Let's examine the extracted fields from the invoice:


In [4]:
# Display comprehensive results
if analysis_result and "result" in analysis_result:
    result = analysis_result["result"]
    contents = result.get("contents", [])
    
    if contents:
        first_content = contents[0]
        
        # Display extracted fields
        fields = first_content.get("fields", {})
        print("Extracted Fields:")
        print("-" * 80)
        if fields:
            for field_name, field_value in fields.items():
                field_type = field_value.get("type")
                if field_type == "string":
                    print(f"{field_name}: {field_value.get('valueString')}")
                elif field_type == "number":
                    print(f"{field_name}: {field_value.get('valueNumber')}")
                elif field_type == "date":
                    print(f"{field_name}: {field_value.get('valueDate')}")
                elif field_type == "array":
                    print(f"{field_name} (array with {len(field_value.get('valueArray', []))} items):")
                    for idx, item in enumerate(field_value.get('valueArray', []), 1):
                        if item.get('type') == 'object':
                            print(f"  Item {idx}:")
                            for key, val in item.get('valueObject', {}).items():
                                if val.get('type') == 'string':
                                    print(f"    {key}: {val.get('valueString')}")
                                elif val.get('type') == 'number':
                                    print(f"    {key}: {val.get('valueNumber')}")
                                # Display confidence and source for nested fields
                                if val.get('confidence') is not None:
                                    print(f"      Confidence: {val.get('confidence'):.3f}")
                                if val.get('source'):
                                    print(f"      Bounding Box: {val.get('source')}")
                elif field_type == "object":
                    print(f"{field_name}: {field_value.get('valueObject')}")
                
                # Display confidence and bounding box for the field
                confidence = field_value.get('confidence')
                if confidence is not None:
                    print(f"  Confidence: {confidence:.3f}")
                source = field_value.get('source')
                if source:
                    print(f"  Bounding Box: {source}")
                print()
        else:
            print("No fields extracted")
        print()
        
        # Display content metadata
        print("Content Metadata:")
        print("-" * 80)
        print(f"Kind: {first_content.get('kind')}")
        if first_content.get("kind") == "document":
            start_page = first_content.get("startPageNumber", 0)
            end_page = first_content.get("endPageNumber", 0)
            print(f"Pages: {start_page} - {end_page}")
            print(f"Total pages: {end_page - start_page + 1}")
        print()
    
    # Save full result to file
    saved_file_path = save_json_to_file(analysis_result, filename_prefix="prebuilt_invoice_analysis_result")
    print(f"Full analysis result saved. Review the complete JSON at: {saved_file_path}")
else:
    print("No analysis result available")



Extracted Fields:
--------------------------------------------------------------------------------
AmountDue: {'Amount': {'type': 'number', 'valueNumber': 610, 'spans': [{'offset': 1523, 'length': 6}], 'confidence': 0.366, 'source': 'D(1,7.4496,8.0513,7.9272,8.0513,7.9272,8.2124,7.4496,8.2124)'}, 'CurrencyCode': {'type': 'string', 'valueString': 'USD'}}

BalanceForward: {'Amount': {'type': 'number', 'valueNumber': 500, 'spans': [{'offset': 1475, 'length': 6}], 'confidence': 0.366, 'source': 'D(1,7.4434,7.7478,7.9272,7.7458,7.9272,7.9070,7.4434,7.9089)'}, 'CurrencyCode': {'type': 'string', 'valueString': 'USD'}}

BillingAddress: 123 Bill St, Redmond WA, 98052
  Confidence: 0.317
  Bounding Box: D(1,0.5801,3.9488,1.2856,3.9470,1.2860,4.1130,0.5805,4.1148);D(1,0.5728,4.1489,2.0254,4.1489,2.0254,4.3167,0.5728,4.3167)

BillingAddressRecipient: Microsoft Finance
  Confidence: 0.377
  Bounding Box: D(1,0.5753,3.7506,1.8028,3.7549,1.8023,3.9115,0.5748,3.9072)

CountryRegion: USA

CustomerAddre

## 2. Receipt Field Extraction with Prebuilt Analyzer

Let's demonstrate using `prebuilt-receipt` to extract structured data from a receipt image. This analyzer automatically identifies merchant information, items, totals, taxes, and payment details without any configuration.


In [5]:
sample_file_path = '../data/receipt.png'
receipt_analyzer_id = "prebuilt-receipt"

print(f"Analyzing {sample_file_path} with {receipt_analyzer_id}...")

analysis_response = client.begin_analyze_binary(
    analyzer_id=receipt_analyzer_id,
    file_location=sample_file_path,
)

# Wait for analysis completion
print(f"Waiting for document analysis to complete...")
analysis_result = client.poll_result(analysis_response)
print(f"Document analysis completed successfully!")



Analyzing ../data/receipt.png with prebuilt-receipt...
Waiting for document analysis to complete...
Document analysis completed successfully!


**Receipt Analysis Results**

Let's examine the extracted fields from the receipt:


In [6]:
# Save the analysis result to a file
saved_file_path = save_json_to_file(analysis_result, filename_prefix="prebuilt_receipt_analysis_result")
# Print the full analysis result as a JSON string
print(json.dumps(analysis_result, indent=2)[:1800])



Analysis result saved to: test_output\prebuilt_receipt_analysis_result_20261106_083537.json
{
  "id": "c855d264-a3d0-4b27-9804-21bc1e306f8f",
  "status": "Succeeded",
  "result": {
    "analyzerId": "prebuilt-receipt",
    "apiVersion": "2025-11-01",
    "createdAt": "2026-11-06T08:35:27Z",
    "stringEncoding": "codePoint",
    "warnings": [],
    "contents": [
      {
        "path": "input1/segment1",
        "category": "receipt.generic",
        "markdown": "Contoso\n\n123 Main Street\nRedmond, WA 98052\n\n987-654-3210\n\n6/10/2019 13:59\nSales Associate: Paul\n\n2 Surface Pro 6\n\n$1,998.00\n\n3 Surface Pen\n\n$299.97\n\nSub-Total\n\n$2,297.97\n\nTax\n\n$218.31\n\nTotal\n\n$2,516.28\n",
        "fields": {
          "MerchantName": {
            "type": "string",
            "valueString": "Contoso",
            "spans": [
              {
                "offset": 0,
                "length": 7
              }
            ],
            "confidence": 0.378,
            "source": 

# Custom Analyzers

Now let's explore creating custom analyzers to extract specific fields tailored to your needs. Custom analyzers allow you to define exactly what information you want to extract and how it should be structured.


**Key Analyzer Configuration Components:**

- **`baseAnalyzerId`**: Specifies which prebuilt analyzer to inherit from. Available base analyzers:
  - **`prebuilt-document`** - For document-based custom analyzers (PDFs, images, Office docs)
  - **`prebuilt-audio`** - For audio-based custom analyzers
  - **`prebuilt-video`** - For video-based custom analyzers
  - **`prebuilt-image`** - For image-based custom analyzers

- **`fieldSchema`**: Defines the structured data to extract from content:
  - **`fields`**: Object defining each field to extract, with field names as keys
  - Each field definition includes:
    - **`type`**: Data type (`string`, `number`, `boolean`, `date`, `object`, `array`)
    - **`description`**: Clear explanation of the field - acts as a prompt to guide extraction accuracy
    - **`method`**: Extraction method to use:
      - **`"extract"`** - Extract values as they appear in content (literal text extraction). Requires `estimateSourceAndConfidence: true`. Only supported for document analyzers.
      - **`"generate"`** - Generate values using AI based on content understanding (best for complex fields)
      - **`"classify"`** - Classify values against predefined categories (use with `enum`)
    - **`enum`**: (Optional) Fixed list of possible values for classification
    - **`items`**: (For arrays) Defines structure of array elements
    - **`properties`**: (For objects) Defines nested field structure

- **`config`**: Processing options that control analysis behavior:
  - **`returnDetails`**: Include confidence scores, bounding boxes, metadata (default: false)
  - **`enableOcr`**: Extract text from images/scans (default: true, document only)
  - **`enableLayout`**: Extract layout info like paragraphs, structure (default: true, document only)
  - **`estimateFieldSourceAndConfidence`**: Return source locations and confidence for extracted fields (document only)
  - **`locales`**: Language codes for transcription (audio/video, e.g., `["en-US"]`)
  - **`contentCategories`**: Define categories for classification and segmentation
  - **`enableSegment`**: Split content into categorized chunks (document/video)

- **`models`**: Specifies which AI models to use:
  - **`completion`**: Model for extraction/generation tasks (e.g., `"gpt-4o"`, `"gpt-4o-mini"`)
  - **`embedding`**: Model for embedding tasks when using knowledge bases

For complete details, see the [Analyzer Reference Documentation](https://learn.microsoft.com/en-us/azure/ai-services/content-understanding/concepts/analyzer-reference).


# Document Analysis

Let's start with document analysis by extracting fields from invoices and receipts. This modality is excellent for processing structured documents and extracting key information like amounts, dates, vendor details, and line items.

## 1. Invoice Field Extraction

Let's extract fields from an invoice PDF. This analyzer identifies essential invoice elements such as vendor information, amounts, dates, and line items.

**Create and Run Invoice Analyzer**

Now let's create the invoice analyzer and process our sample invoice:

In [7]:
import time
invoice_analyzer_id = f"notebooks_sample_invoice_extraction_{int(time.time())}"

invoice_analyzer = {
    "baseAnalyzerId": "prebuilt-document",
    "description": "Sample invoice analyzer that extracts vendor information, line items, and totals from commercial invoices",
    "config": {
        "returnDetails": True,
        "enableOcr": True,
        "enableLayout": True,
        "estimateFieldSourceAndConfidence": True
    },
    "fieldSchema": {
        "name": "InvoiceFields",
        "fields": {
            "VendorName": {
                "type": "string",
                "method": "extract",
                "description": "Name of the vendor or supplier, typically found in the header section"
            },
            "Items": {
                "type": "array",
                "method": "generate",
                "description": "List of items or services on the invoice, typically in a table format",
                "items": {
                    "type": "object",
                    "properties": {
                        "Description": {
                            "type": "string",
                            "description": "Item or service description"
                        },
                        "Amount": {
                            "type": "number",
                            "description": "Line total amount for this item"
                        }
                    }
                }
            }
        }
    },
    "models": {
        "completion": COMPLETION_MODEL
    }
}
print(f"{json.dumps(invoice_analyzer, indent=2)}")
# Start the analyzer creation operation
response = client.begin_create_analyzer(
    analyzer_id=invoice_analyzer_id,
    analyzer_template=invoice_analyzer,
)

# Wait for the analyzer to be created
print(f"Waiting for analyzer creation to complete...")
client.poll_result(response)
print(f"Analyzer '{invoice_analyzer_id}' created successfully!")


{
  "baseAnalyzerId": "prebuilt-document",
  "description": "Sample invoice analyzer that extracts vendor information, line items, and totals from commercial invoices",
  "config": {
    "returnDetails": true,
    "enableOcr": true,
    "enableLayout": true,
    "estimateFieldSourceAndConfidence": true
  },
  "fieldSchema": {
    "name": "InvoiceFields",
    "fields": {
      "VendorName": {
        "type": "string",
        "method": "extract",
        "description": "Name of the vendor or supplier, typically found in the header section"
      },
      "Items": {
        "type": "array",
        "method": "generate",
        "description": "List of items or services on the invoice, typically in a table format",
        "items": {
          "type": "object",
          "properties": {
            "Description": {
              "type": "string",
              "description": "Item or service description"
            },
            "Amount": {
              "type": "number",
              

Let's run the custom analyzer with a invoice pdf.

In [8]:
sample_file_path = '../data/invoice.pdf'

# Begin document analysis operation
print(f"Starting document analysis with analyzer '{invoice_analyzer_id}'...")
analysis_response = client.begin_analyze_binary(
    analyzer_id=invoice_analyzer_id,
    file_location=sample_file_path,
)

# Wait for analysis completion
print(f"Waiting for document analysis to complete...")
analysis_result = client.poll_result(analysis_response)
print(f"Document analysis completed successfully!")



Starting document analysis with analyzer 'notebooks_sample_invoice_extraction_1793954137'...
Waiting for document analysis to complete...
Document analysis completed successfully!


**Invoice Analysis Results**

Let's examine the extracted fields from the invoice:

In [9]:
# Display comprehensive results
if analysis_result and "result" in analysis_result:
    result = analysis_result["result"]
    contents = result.get("contents", [])
    
    if contents:
        first_content = contents[0]
        
        # Display extracted fields
        fields = first_content.get("fields", {})
        print("Extracted Fields:")
        print("-" * 80)
        if fields:
            for field_name, field_value in fields.items():
                field_type = field_value.get("type")
                if field_type == "string":
                    print(f"{field_name}: {field_value.get('valueString')}")
                elif field_type == "number":
                    print(f"{field_name}: {field_value.get('valueNumber')}")
                elif field_type == "array":
                    print(f"{field_name} (array with {len(field_value.get('valueArray', []))} items):")
                    for idx, item in enumerate(field_value.get('valueArray', []), 1):
                        if item.get('type') == 'object':
                            print(f"  Item {idx}:")
                            for key, val in item.get('valueObject', {}).items():
                                if val.get('type') == 'string':
                                    print(f"    {key}: {val.get('valueString')}")
                                elif val.get('type') == 'number':
                                    print(f"    {key}: {val.get('valueNumber')}")
                elif field_type == "object":
                    print(f"{field_name}: {field_value.get('valueObject')}")
                print()
        else:
            print("No fields extracted")
        print()
        
        # Display content metadata
        print("Content Metadata:")
        print("-" * 80)
        print(f"Kind: {first_content.get('kind')}")
        print(f"Pages: {first_content.get('startPageNumber')} - {first_content.get('endPageNumber')}")
        print(f"Unit: {first_content.get('unit')}")
        print()
        
    
    # Save full result to file
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    output_file = f"test_output/invoice_analysis_result_{timestamp}.json"
    os.makedirs("test_output", exist_ok=True)
    
    with open(output_file, 'w') as f:
        json.dump(analysis_result, f, indent=2)
    
    print(f"Full analysis result saved to: {output_file}")
else:
    print("No analysis result available")


Extracted Fields:
--------------------------------------------------------------------------------
VendorName: CONTOSO LTD.

Items (array with 3 items):
  Item 1:
    Description: Consulting Services
    Amount: 60
  Item 2:
    Description: Document Fee
    Amount: 30
  Item 3:
    Description: Printing Fee
    Amount: 10


Content Metadata:
--------------------------------------------------------------------------------
Kind: document
Pages: 1 - 1
Unit: inch

Full analysis result saved to: test_output/invoice_analysis_result_20261106_033548.json


**Clean Up Invoice Analyzer**

Clean up the analyzer to manage resources (in production, you would typically keep analyzers for reuse):

In [10]:
if RUN_SAMPLE_CLEANUP:
    # Clean up the created analyzer
    print(f"Deleting analyzer '{invoice_analyzer_id}'...")
    client.delete_analyzer(analyzer_id=invoice_analyzer_id)
    print(f"Analyzer '{invoice_analyzer_id}' deleted successfully!")
else:
    print('Sample analyzer cleanup skipped. IDs are retained for review; set RUN_SAMPLE_CLEANUP=True explicitly to delete only these sample analyzers.')



Sample analyzer cleanup skipped. IDs are retained for review; set RUN_SAMPLE_CLEANUP=True explicitly to delete only these sample analyzers.


# Summary

**Congratulations!** You've successfully completed the field extraction tutorial for Azure AI Content Understanding!


## Next Steps

- **Try Other Notebooks**: 
  - `content_extraction.ipynb` - Multi-modal content extraction (audio, video, images)
  - `conversational_field_extraction.ipynb` - Extract fields from audio conversations
  - `management.ipynb` - Advanced analyzer management operations
- **Read the Documentation**: Visit the [Azure AI Content Understanding documentation](https://learn.microsoft.com/azure/ai-services/content-understanding/) for comprehensive guides and API references